# Overview
The goal of this study is to evaluate and compare different clustering algorithms.
it consists of 4 key parts:
1. Loading datasets
2. Clustering with each clustering algorithm
3. Measuring performance
4. Summary


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import clustbench
import genieclust 
import pandas as pd


# Loading datasets

In [ ]:
from utils.loader import load_datasets
datasets = load_datasets()


Loading datasets from graves: 100%|██████████| 10/10 [00:00<00:00, 598.84it/s]


### Algorithms 
following clustering algorithms will be evaluated: 
- **K-means**

        sklearn.cluster.KMeans(n_clusters)

- **Gaussian mixture model**

        sklearn.mixture.GaussianMixture(n_components)

- **Genie** (with parameter $g\in \{0.1,0.3,0.5,0.7,0.9\}$

        genieclust.Genie(n_clusters, gini_threshold = g)

- **Agglomerative Hierarchical Clustering** (with single, average, complete, and Ward linkage)

        sklearn.cluster.AgglomerativeClustering(n_clusters, linkage = linkage)

- **Spectral clustering**

        sklearn.cluster.SpectralClustering(n_clusters)

- **DBSCAN is considered separetely

        sklearn.cluster.DBSCAN(eps)


# Performing clustering

In [84]:
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from genieclust import Genie
from functools import partial

model_configs = [
    ("KMeans", KMeans),
    ("Genie_g0.1", partial(Genie, 
                    gini_threshold=0.1)),
    ("Genie_g0.3", partial(Genie, 
                    gini_threshold=0.3)),
    ("Genie_g0.5", partial(Genie, 
                    gini_threshold=0.5)),
    ("Genie_g0.7", partial(Genie, gini_threshold=0.7)),
    ("Genie_g0.9", partial(Genie, gini_threshold=0.9)),
    ("AHC_ward", partial(AgglomerativeClustering, 
                    linkage='ward')),
    ("AHC_single", partial(AgglomerativeClustering,
                     linkage='single')),
    ("AHC_average", partial(AgglomerativeClustering, 
                    linkage='average')),
    ("AHC_complete", partial(AgglomerativeClustering, 
                    linkage='complete')),
    ("SpectralClustering", partial(SpectralClustering, 
                            eigen_solver='arpack', 
                            affinity='nearest_neighbors', 
                            n_init=1,
                            random_state=42)),
    ("GaussianMixture", GaussianMixture)
]

In [105]:
from utils.clustering import  cluster_datasets_

for model_name, model in model_configs:
    cluster_datasets_(preprocessed_datasets, model, model_name)

Clustering datasets with SpectralClustering:   0%|          | 0/43 [00:00<?, ?it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering datasets with SpectralClustering:   2%|▏         | 1/43 [00:00<00:04,  9.15it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering datasets with SpectralClustering:   5%|▍         | 2/43 [00:00<00:10,  3.85it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering dataset

In [ ]:
import tqdm
from utils.clustering import cluster_datasets
results = {}
for model_name, alg in model_configs:
    model_results = cluster_datasets(datasets, alg, model_name)
    results[model_name] = model_results

Clustering datasets with SpectralClustering:   0%|          | 0/43 [00:00<?, ?it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering datasets with SpectralClustering:   2%|▏         | 1/43 [00:00<00:04,  8.63it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering datasets with SpectralClustering:   5%|▍         | 2/43 [00:00<00:12,  3.36it/s]/Users/Jacek/PythonProjects/DEV/Clustering_Comparison/.venv/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:324: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
Clustering dataset

Widać, ze K-Means i Genie są najszybsze


# Evaluating Created Clusters

### Methodology
For evaluation of clustering algorithms following measures will be used:

- Adjusted Rank Index (ARI)

    $$ARI = (RI - Expected_{RI}) / (\max(RI) - Expected_{RI})$$

    The Rand Index (RI) computes a similarity measure between two clusterings by considering all pairs of samples and counting pairs that are assigned in the same or different clusters in the predicted and true clusterings.https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html

            sklearn.metrics.adjusted_rand_score



- Normalised Clustering Accuracy (NCA)


    NCA is the averaged percentage of correctly classified points in each cluster above the perfectly uniform label distribution. Note that the matching between the cluster labels is performed automatically by finding the best permutation $\sigma$ 

            clustbench.get_score


- Normalised Mutual Information (NMI)

    NMI is a normalization of the Mutual Information (MI) score to scale the results between 0 (no mutual information) and 1 (perfect correlation). In this function, mutual information is normalized by some generalized mean of H(labels_true) and H(labels_pred)), defined by the

            sklearn.metrics.normalized_mutual_info_score


In [88]:
dataset_Xs = {}
for dataset in datasets:
    print('dataset:', dataset.name)
    data =  clustbench.preprocess_data(dataset.data)
    mask = dataset.labels[0]!=0
    X = data[mask]
    dataset_Xs[dataset.name] = X
    

for model_name, model_results in results.items():
    for dataset_name, metrics in model_results.items():
        ...

AttributeError: 'tuple' object has no attribute 'name'

## DBSCAN


Due to its nature DBSCAN will be considered separately 


In [ ]:
def analyze_dbscan_noise(y_true, y_dbscan):
    # Points that are noise in the ground truth (label: 0)
    gt_noise_mask = (y_true == 0)
    # Points that are noise in the DBSCAN results (label: -1)
    db_noise_mask = (y_dbscan == -1)
    
    stats = {
        "total_points": len(y_true),
        "gt_noise_count": np.sum(gt_noise_mask),
        "db_noise_count": np.sum(db_noise_mask),
        # TP for noise
        "correctly_detected_noise": np.sum(gt_noise_mask & db_noise_mask),
        # FP for noise
        "false_noise_detection": np.sum(~gt_noise_mask & db_noise_mask)
    }
    return stats

# SUMMARY